# Urban Safety — Risk Score Calculation

Computes a normalized risk score from the filtered feature dataset, classifies segments into risk bands, and exports the scored output.

**Input:** `csv/features_all_boroughs_with_location_id_augmented.csv`  
**Output:** `csv/segment_risk_scores_w-id.csv`

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(rc={'figure.figsize': (8, 8)})
pd.set_option('display.float_format', '{:.3f}'.format)

FEATURES_CACHE = os.path.join('..', 'csv', 'features_all_boroughs_with_location_id_augmented.csv')
OUTPUT_PATH = os.path.join('..', 'csv', 'segment_risk_scores_w-id.csv')

ALLOWED_BOROUGH_NAMES = [
    'City of Westminster, London, UK',
    'London Borough of Islington, London, UK',
    'London Borough of Hackney, London, UK',
    'London Borough of Tower Hamlets, London, UK',
    'London Borough of Southwark, London, UK',
]

FEATURE_COLS = ['lighting', 'visibility', 'connectivity', 'enclosure', 'public_transport_proximity_m', 'dominant_land_use_score']

## Load Filtered Features
Loads the prepared feature CSV and keeps only the five target boroughs.

In [ ]:
features = pd.read_csv(FEATURES_CACHE)
if 'borough' in features.columns:
    before = len(features)
    features = features[features['borough'].isin(ALLOWED_BOROUGH_NAMES)].reset_index(drop=True)
    after = len(features)
    print(f'✓ Loaded {after:,} segments from {FEATURES_CACHE} (filtered from {before:,})')
    print(f'  Boroughs : {features["borough"].nunique()}')
    print(features['borough'].value_counts().to_string())
else:
    print(f'✓ Loaded {len(features):,} segments from {FEATURES_CACHE} (no borough column to filter)')

## Risk Score Formula
The score is normalized to the $[0, 1]$ range after combining the weighted feature components.

In [ ]:
def compute_risk_score(df):
    def norm(col):
        col = col.astype(float)
        min_val = col.min()
        max_val = col.max()
        if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
            return pd.Series(0.0, index=col.index)
        return (col - min_val) / (max_val - min_val)

    lighting = norm(df['lighting'].fillna(df['lighting'].median()))
    connectivity = norm(df['connectivity'].fillna(df['connectivity'].median()))
    enclosure = norm(df['enclosure'].fillna(df['enclosure'].median()))
    visibility = norm(df['visibility'].fillna(df['visibility'].median()))

    if 'public_transport_proximity_m' in df.columns:
        transport_raw = df['public_transport_proximity_m']
    elif {'dist_to_rail_m', 'dist_to_bus_m'}.issubset(df.columns):
        transport_raw = df[['dist_to_rail_m', 'dist_to_bus_m']].min(axis=1)
    else:
        transport_raw = pd.Series(0.0, index=df.index)
    transport = 1 - norm(transport_raw.fillna(transport_raw.median()).clip(upper=800))

    if 'dominant_land_use_score' in df.columns:
        landuse_raw = df['dominant_land_use_score']
    elif 'commercial_ratio' in df.columns:
        landuse_raw = df['commercial_ratio']
    else:
        landuse_raw = pd.Series(0.0, index=df.index)
    landuse = norm(landuse_raw.fillna(landuse_raw.median()))

    score = (
        -0.25 * lighting
        +0.15 * enclosure
        -0.15 * visibility
        -0.15 * connectivity
        -0.15 * transport
        -0.05 * landuse
    )

    if score.max() == score.min():
        return pd.Series(0.0, index=df.index)
    return (score - score.min()) / (score.max() - score.min())

## Classification
Converts the normalized score into low, medium, and high risk classes.

In [ ]:
features['risk_score'] = compute_risk_score(features)

threshold_low_med = 0.33
threshold_med_high = 0.66

features['risk_class'] = pd.cut(
    features['risk_score'],
    bins=[0, threshold_low_med, threshold_med_high, 1.0],
    labels=['low', 'medium', 'high'],
    include_lowest=True
)
features['risk_class'] = pd.Categorical(
    features['risk_class'],
    categories=['low', 'medium', 'high'],
    ordered=True
)
features['safety_class'] = features['risk_class']

print('Risk class thresholds (fixed-score bins):')
print(f'  low    : 0 to {threshold_low_med:.2f}')
print(f'  medium : {threshold_low_med:.2f} to {threshold_med_high:.2f}')
print(f'  high   : {threshold_med_high:.2f}+')
print(f'\nActual distribution:')
for label in ['low', 'medium', 'high']:
    mask = features['risk_class'] == label
    if mask.sum() > 0:
        min_val = features.loc[mask, 'risk_score'].min()
        max_val = features.loc[mask, 'risk_score'].max()
        pct = 100 * mask.sum() / len(features)
        print(f'  {label:10s} : {min_val:8.4f} to {max_val:8.4f} ({mask.sum():6,} segments, {pct:5.1f}%)')

## Export
Writes the scored features to a CSV for downstream analysis.

In [ ]:
if 'location_id' not in features.columns:
    if 'osmid' in features.columns:
        features['location_id'] = features['borough'].str.extract(r'(\w+)')[0] + '_' + features['osmid'].astype(str)
    else:
        features['location_id'] = features['borough'].str.extract(r'(^[A-Za-z]+)')[0].str.upper() + '_' + features.index.astype(str)

segment_scores = features[['borough', 'lighting', 'visibility', 'connectivity', 'enclosure',
                            'public_transport_proximity_m', 'dominant_land_use_score',
                            'risk_score', 'risk_class', 'location_id']].copy()

segment_scores.to_csv(OUTPUT_PATH, index=False)
print(f'✓ Exported {len(segment_scores):,} segments')
print(f'✓ Saved to {OUTPUT_PATH}')
display(segment_scores.head(10))

## Validation
Check the risk score distribution and the class balance.

In [ ]:
print('=== Risk Score Summary ===\n')
print(f'Min risk_score: {features["risk_score"].min():.4f}')
print(f'Max risk_score: {features["risk_score"].max():.4f}')
print(f'Mean risk_score: {features["risk_score"].mean():.4f}')
print(f'Median risk_score: {features["risk_score"].median():.4f}')
print(f'\nRisk class distribution:')
print(features['risk_class'].value_counts().sort_index())

## Risk Plots
Visual checks for the risk score distribution.

In [ ]:
os.makedirs('plots', exist_ok=True)

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(features['risk_score'], bins=80, color='steelblue', alpha=0.7, edgecolor='none', label='Histogram')
ax.set_xlabel('Risk Score')
ax.set_ylabel('Frequency (log scale)')
ax.set_yscale('log')
ax.set_title('Distribution of Risk Scores Across All Segments')
ax.axvline(threshold_low_med, color='black', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Low->Medium ({threshold_low_med:.2f})')
ax.axvline(threshold_med_high, color='black', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Medium->High ({threshold_med_high:.2f})')
ax.set_xlim([features['risk_score'].min() - 0.05, features['risk_score'].max() + 0.05])
ax.legend(); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('plots/risk_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
features_copy = features.copy()
features_copy['borough_short'] = (features_copy['borough']
    .str.replace('London Borough of ', '').str.replace(', London, UK', '').str.replace('City of ', ''))
features_copy.boxplot(column='risk_score', by='borough_short', ax=ax)
ax.set_xlabel('Borough'); ax.set_ylabel('Risk Score')
ax.set_title('Risk Score Distribution by Borough')
plt.suptitle(''); plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plots/risk_score_by_borough.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(12, 10))
class_counts = features['risk_class'].value_counts().reindex(['low', 'medium', 'high'])
colors_list = ['#C8E6C9', '#FFF9C4', '#FFCCBC']
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors_list, edgecolor='black',
                   linewidth=2, alpha=0.85)
for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height, f'{int(height):,}',
                 ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].set_ylabel('Number of Segments', fontsize=12, fontweight='bold')
axes[0].set_title('Segments by Risk Class', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y', linestyle='--')
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].hist(features['risk_score'], bins=50, color='steelblue', edgecolor='black',
             linewidth=0.5, alpha=0.7)
axes[1].set_xlabel('Risk Score', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Segments', fontsize=12, fontweight='bold')
axes[1].set_title('Risk Score Distribution', fontsize=12, fontweight='bold')
axes[1].axvline(threshold_low_med, color='red', linestyle='--', linewidth=2, alpha=0.7,
                label=f'Low->Medium ({threshold_low_med:.2f})')
axes[1].axvline(threshold_med_high, color='darkred', linestyle='--', linewidth=2, alpha=0.7,
                label=f'Medium->High ({threshold_med_high:.2f})')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y', linestyle='--')
axes[1].spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('plots/risk_score_by_class.png', dpi=150, bbox_inches='tight')
plt.show()